# Embase 2026 — email-registration tickets, and the contact reason for everything else

Source: `data/embase/embase verbstim extraction_2026-07-27_todate.tsv` — the raw thread export,
**one row per message**, not per ticket. Same layout as the LeapSpace export handled by notebook 18,
and cleaned by the same code.

| stage | module | runs on | decides |
|---|---|---|---|
| A | `embase_emailreg` | every ticket | is this ticket about email registration / an email-account issue? |
| B | `embase_emailreg_sub` | stage-A positives | which sub-category of email-account issue |
| C | `embase_contact_reason` | stage-A negatives | the main contact reason |

There is no Embase taxonomy in the repo, so **both B and C are derived from the conversations**:
`rca build discover` drafts each one, the draft is curated by hand after reading the
tickets, and classification only starts once the curated lists have been reviewed (section 8).

The companion `embase metadata extraction_2026-07-27_todate.csv` carries the contact reason the
**agent** picked for each ticket. It is not used to classify anything; it is joined onto the
deliverable as `agent_cr_l1` / `agent_cr_l2` so the derived contact reason can be read against it.

## What the profiling found

- **3,037 message rows over 774 tickets**, 2025-12-04 to 2026-09-29. `INCIDENT_ID` /
  `INCIDENT_REF` are 1:1.
- **266 rows are exact duplicates** (same `THREAD_ID`, identical in every column). Dropped.
- **`sequence` is not a message order** — it is 1 on 90% of rows, and in 211 of the 263 tickets
  where it varies it contradicts the timestamps. Messages are ordered by `DATE_CREATED`.
- **Five channels, one per ticket:** `Web` (414), `CSS Email` (282), `Chat` (56), `Phone` (21),
  `Fax` (1). Every message is its own row whatever the channel, so the rebuild is the same.
- **Sender is `THREAD_ENTRY_TYPE`:** `Customer`, `Staff Account` (the agent), and `Customer Proxy`
  — a note the agent typed on the customer's behalf, labelled `CUSTOMER (logged by agent)`.
- **`VERBATIM` is mixed plain text and Outlook/Word HTML**, with quoted reply history. Quoted
  history is cut from replies but not from a ticket's opening message, where it is often a
  forwarded customer request.
- **The metadata file has 782 rows**, covering all 774 tickets (8 extra rows have no thread).
  Its `contact_reason_lvl1_desc` / `contact_reason_lvl2_desc` names collide with the pipeline's
  structural prefixes and would be silently stripped, so they are renamed on join.

## 0. Configuration

In [ ]:
import hashlib
import html
import json
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from html.parser import HTMLParser
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
SRC_TSV      = BASE / 'data' / 'embase' / 'embase verbstim extraction_2026-07-27_todate.tsv'
METADATA_CSV = BASE / 'data' / 'embase' / 'embase metadata extraction_2026-07-27_todate.csv'
for _p in (SRC_TSV, METADATA_CSV):
    assert _p.exists(), f'missing input: {_p}'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'embase_2026_email_registration_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
DRAFT_DIR     = TAXONOMY_DIR / 'drafts'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_embase_emailreg'

# -- Columns ---------------------------------------------------------------
ID_COL   = 'INCIDENT_REF'
TEXT_COL = 'Conversation'

# -- Modules ---------------------------------------------------------------
MOD_G, PREFIX_G, STUB_G = 'embase_emailreg',       'EER', 'eer'   # stage A gate
MOD_S, PREFIX_S, STUB_S = 'embase_emailreg_sub',   'EES', 'ees'   # stage B sub-category
MOD_C, PREFIX_C, STUB_C = 'embase_contact_reason', 'ECR', 'ecr'   # stage C contact reason
MODULE_PREFIXES = {MOD_G: PREFIX_G, MOD_S: PREFIX_S, MOD_C: PREFIX_C}
ALL_MODULES = list(MODULE_PREFIXES)

# -- Preprocessing --------------------------------------------------------
TRUNC_CAP  = 16_000
TRUNC_HEAD = 4_000
TRUNC_TAIL = 11_900

# -- LLM tuning -----------------------------------------------------------
# batch_size 1: long evidence prompts batched together have swapped answers
# between tickets before.
TUNING = {
    'batch_size': 1, 'max_chars': 40_000, 'concurrency': 4,
    'retries': 5, 'max_tokens': 4096, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
    'temperature': 0.0,
}
OUTPUT_AFFECTING = ('temperature', 'max_tokens', 'batch_size', 'max_chars')
assert set(OUTPUT_AFFECTING) <= set(TUNING)

# -- Outputs --------------------------------------------------------------
INTER_DIR = BASE / 'runs' / 'intermediate'
SMOKE_DIR = BASE / 'runs' / 'smoke_embase_emailreg'
OUT_DIR   = BASE / 'runs' / 'output'
for _d in (INTER_DIR, SMOKE_DIR, OUT_DIR, DEFN_XLSX.parent, BASE / 'profiles'):
    _d.mkdir(parents=True, exist_ok=True)

CONV_CSV  = INTER_DIR / 'embase_emailreg_conversations.csv'   # deliverable: cleaned threads
INPUT_G   = INTER_DIR / 'embase_emailreg_gate_input.csv'
INPUT_S   = INTER_DIR / 'embase_emailreg_sub_input.csv'
INPUT_C   = INTER_DIR / 'embase_emailreg_cr_input.csv'
PROFILE_G = BASE / 'profiles' / 'embase_2026_emailreg_gate.json'
PROFILE_S = BASE / 'profiles' / 'embase_2026_emailreg_sub.json'
PROFILE_C = BASE / 'profiles' / 'embase_2026_emailreg_cr.json'
OUT_G     = OUT_DIR / 'embase_2026_emailreg_gate.xlsx'
OUT_S     = OUT_DIR / 'embase_2026_emailreg_sub.xlsx'
OUT_C     = OUT_DIR / 'embase_2026_emailreg_contact_reason.xlsx'
FINAL_XLSX = OUT_DIR / 'embase_2026_email_registration.xlsx'
FINAL_CSV  = OUT_DIR / 'embase_2026_email_registration_full_text.csv'

SUBPROCESS_ENV = {**os.environ, 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}

print(f'Project root : {BASE}')
print(f'Tickets      : {SRC_TSV.relative_to(BASE)}')
print(f'Metadata     : {METADATA_CSV.relative_to(BASE)}')
print(f'Definition   : {DEFN_XLSX.relative_to(BASE)}')
print(f'Modules      : {ALL_MODULES}')

## 0b. Shared helpers

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '\u200b\u200c\u200d\u2060\ufeff\u00ad'))


def norm_label(value) -> str:
    """Normalise a label to a stable join key, preserving its wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = unicodedata.normalize('NFKC', text.translate(_ZERO_WIDTH))
    return re.sub(r'\s+', ' ', text).strip()


def leaf_labels(module: str, tax_dir: Path = TAXONOMY_DIR) -> dict:
    """code -> label for every leaf of a taxonomy JSON."""
    tax = json.loads((tax_dir / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label, ok_codes=(0,)):
    """Run a short command, echoing its output; raise on an unexpected exit code."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode not in ok_codes:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')
    return r


print('Helpers defined.')

## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook only,
so whatever product is live right now is dropped. It is backed up first, with the command that puts
it back written alongside.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_live = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
_live_modules = [m for m in _live['modules'] if m not in ALL_MODULES]

_backed_up = []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *[TAXONOMY_DIR / f'{m}.json' for m in _live_modules]]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():                 # never clobber an earlier backup
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}')
for _n in _backed_up:
    print(f'  {_n}')
print(f'\nModules in the live config right now: {list(_live["modules"])}')

_DEFN_OF = {'s2ls_': 'scopus_2ls_2026_pipeline_definition.xlsx',
            'ls_emailreg': 'leapspace_2026_email_registration_definition.xlsx',
            'regid_': 'reg_id_2026_pipeline_definition.xlsx',
            'sd_ac_': 'sd_2026_accessing_content_definition.xlsx',
            'sd_': 'sd_2026_pipeline_definition.xlsx',
            'csd_': 'csd_2026_pipeline_definition.xlsx',
            'leapspace': 'leapspace_2026_pipeline_definition.xlsx'}
_defn = next((v for k, v in _DEFN_OF.items()
              if any(m.startswith(k) for m in _live_modules)), None)
_restore = (f'rca build --input definitions/{_defn} --force-all'
            if _defn else '(no known definition workbook rebuilds these - see generated/)')

# Written once. On a re-run the live config is already this notebook's own, and
# overwriting would lose the only record of what was there before.
RESTORE_TXT = CONFIG_BACKUP / 'RESTORE.txt'
if RESTORE_TXT.exists():
    print(f'\n{RESTORE_TXT.relative_to(BASE)} already records the pre-run state; left untouched:\n')
    print('    ' + RESTORE_TXT.read_text(encoding='utf-8').replace('\n', '\n    '))
else:
    RESTORE_TXT.write_text(
        'The modules that were live in generated/pipeline_config.json before this notebook ran:\n'
        f'    {", ".join(_live_modules) or "(none)"}\n\n'
        'To put them back:\n\n'
        f'    {_restore}\n\n'
        'The config holds one product at a time; restoring drops the embase_* modules.\n',
        encoding='utf-8')
    print(f'\nTo restore afterwards:\n    {_restore}')

## 1. Load and de-duplicate the messages

In [ ]:
EXPECTED_COLS = ['INCIDENT_ID', 'INCIDENT_CHANNEL', 'AGENT_ID', 'CONTACT_ID', 'DATE_CREATED',
                 'THREAD_ID', 'sequence', 'VERBATIM', 'THREAD_ENTRY_TYPE', 'INCIDENT_TYPE',
                 'INCIDENT_REF']
# keep_default_na=False: a customer row has an empty AGENT_ID, not NaN, and a
# VERBATIM of "NA" is text.
raw = pd.read_csv(SRC_TSV, sep='\t', dtype=str, encoding='utf-8', keep_default_na=False)
assert list(raw.columns) == EXPECTED_COLS, f'columns changed: {list(raw.columns)}'
print(f'{SRC_TSV.name}: {len(raw):,} rows, {raw["INCIDENT_ID"].nunique():,} tickets')

_n_dup = int(raw.duplicated().sum())
msgs = raw.drop_duplicates().copy()
assert msgs['THREAD_ID'].is_unique, 'a THREAD_ID repeats with different content - inspect before dropping'
print(f'Exact duplicate rows dropped: {_n_dup:,}  ->  {len(msgs):,} messages')

# One ticket = one ref, one channel, one type. The conversation rebuild relies on it.
for _c in ('INCIDENT_REF', 'INCIDENT_CHANNEL', 'INCIDENT_TYPE'):
    _n = msgs.groupby('INCIDENT_ID')[_c].nunique().max()
    assert _n == 1, f'{_c} varies within a ticket (max {_n} values)'
assert msgs.groupby('INCIDENT_REF')['INCIDENT_ID'].nunique().max() == 1

SPEAKER = {'Customer': 'CUSTOMER', 'Staff Account': 'AGENT',
           'Customer Proxy': 'CUSTOMER (logged by agent)'}
CHANNEL_NAME = {'Web': 'Web', 'CSS Email': 'Email', 'Chat': 'Chat', 'Phone': 'Phone',
                'Fax': 'Fax'}
assert set(msgs['THREAD_ENTRY_TYPE']) <= set(SPEAKER), \
    f'unknown sender types: {set(msgs["THREAD_ENTRY_TYPE"]) - set(SPEAKER)}'
assert set(msgs['INCIDENT_CHANNEL']) <= set(CHANNEL_NAME), \
    f'unknown channels: {set(msgs["INCIDENT_CHANNEL"]) - set(CHANNEL_NAME)}'
# The customer/agent split is also visible in AGENT_ID - check the two agree.
_cust = msgs['THREAD_ENTRY_TYPE'] == 'Customer'
assert (msgs.loc[_cust, 'AGENT_ID'] == '').all(), 'a Customer row carries an AGENT_ID'
assert (msgs.loc[~_cust, 'AGENT_ID'] != '').all(), 'a Staff/Proxy row has no AGENT_ID'

msgs['ts'] = pd.to_datetime(msgs['DATE_CREATED'], format='%Y-%m-%d %H:%M:%S.%f')
print(f'Dates: {msgs["ts"].min():%Y-%m-%d} to {msgs["ts"].max():%Y-%m-%d}')
print(f'sequence == 1 on {msgs["sequence"].eq("1").mean():.0%} of rows - not an order; using DATE_CREATED\n')
print(pd.crosstab(msgs['INCIDENT_CHANNEL'], msgs['THREAD_ENTRY_TYPE'], margins=True).to_string())
print('\nTickets per channel:')
print(msgs.groupby('INCIDENT_CHANNEL')['INCIDENT_ID'].nunique().to_string())

### The agent's contact reason, for reference only

Renamed on the way in: `contact_reason_lvl1_desc` and `resolution_code_lvl1_desc` start with
prefixes the pipeline treats as its own and strips.

In [ ]:
meta = pd.read_csv(METADATA_CSV, dtype=str, keep_default_na=False)
assert meta['INCIDENT_ID'].is_unique, 'metadata has more than one row per ticket'
_missing = set(msgs['INCIDENT_ID']) - set(meta['INCIDENT_ID'])
assert not _missing, f'{len(_missing)} tickets have no metadata row'
print(f'{METADATA_CSV.name}: {len(meta)} rows; '
      f'{len(set(meta["INCIDENT_ID"]) - set(msgs["INCIDENT_ID"]))} have no thread in the TSV (ignored)')

META_COLS = {'incident_subject': 'incident_subject',
             'contact_reason_lvl1_desc': 'agent_cr_l1',
             'contact_reason_lvl2_desc': 'agent_cr_l2',
             'resolution_code_lvl1_desc': 'agent_resolution',
             'customer_classification_role': 'customer_role',
             'status_desc': 'status'}
meta = meta[['INCIDENT_ID', *META_COLS]].rename(columns=META_COLS)
print('\nAgent contact reason (lvl1 / lvl2), tickets in this export:')
print(meta[meta['INCIDENT_ID'].isin(msgs['INCIDENT_ID'])]
      .groupby(['agent_cr_l1', 'agent_cr_l2']).size().sort_values(ascending=False).to_string())

## 2. Clean each message

Four steps, in order:

1. **`html_to_text`** — block tags (`p`, `div`, `br`, `li`, `tr`, ...) become line breaks, `style`
   and `script` are dropped, entities are decoded. Newlines in HTML *source* are layout and are
   discarded first. A second pass removes tags that were escaped twice in the source
   (`&lt;img ...&gt;`), matched by tag name so that `<user1@example.com>` address brackets survive.
2. **`normalise_ws`** — zero-width characters removed, NFKC (folds `&nbsp;` into a space), runs of
   spaces collapsed, every line trimmed, 3+ blank lines reduced to one.
3. **`strip_quoted_history`** — cut at the first reply header: Outlook `From: … Sent:`/`Date:` (and
   the Dutch, German, French, Spanish, Portuguese, Polish and Chinese forms seen in this data),
   `On … wrote:` (and `escreveu`, `a écrit`, `写道`, ...), `-----Original Message-----`, `原始邮件`.
   **Skipped for the first message of a ticket**, where the quoted part is a forwarded customer
   request. A message that is nothing *but* a quote is kept whole.
4. **`strip_boilerplate`** — the "External email / Trusted partners" banner, `CAUTION: External …`
   lines and the "Visit Elsevier Support Center" signature line. Agent names and sign-offs stay —
   they are part of who said what.
5. **`strip_repeats`** — some clients paste the previous message back in with no reply header at
   all. Any earlier message of the same ticket (80+ characters) found verbatim inside a later one
   is removed from the later one. An exact resend is left alone.

In [ ]:
_BLOCK_TAGS = {'p', 'div', 'br', 'li', 'tr', 'ul', 'ol', 'table', 'h1', 'h2', 'h3', 'h4', 'h5',
               'h6', 'blockquote', 'hr', 'pre'}
_SKIP_TAGS = {'style', 'script', 'head', 'title'}
# A tag name followed by attributes or '>' - so <user2@example.com> in plain text is not HTML.
_HAS_TAG = re.compile(r'<\s*/?\s*[a-zA-Z][a-zA-Z0-9:]*(?:\s[^<>]*)?/?>')
# Tags that survive one pass because the source escaped them twice. The lookahead
# after the name is what keeps <user3@example.com> from being read as an <a> tag.
_LEFTOVER_TAG = re.compile(
    r'</?\s*(?:a|img|div|span|p|br|font|table|tbody|thead|tr|td|th|ul|ol|li|b|i|u|strong|em|'
    r'o:p|hr|h[1-6]|blockquote|meta|style|link|center|sup|sub|col|colgroup)(?=[\s/>])[^<>]*>',
    re.I)


class _TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.parts, self._skip = [], 0

    def handle_starttag(self, tag, attrs):
        if tag in _SKIP_TAGS:
            self._skip += 1
        elif tag in _BLOCK_TAGS:
            self.parts.append('\n')
        if tag == 'li':
            self.parts.append('- ')
        elif tag in ('td', 'th'):
            self.parts.append(' ')

    def handle_startendtag(self, tag, attrs):
        if tag in _BLOCK_TAGS:
            self.parts.append('\n')

    def handle_endtag(self, tag):
        if tag in _SKIP_TAGS:
            self._skip = max(0, self._skip - 1)
        elif tag in _BLOCK_TAGS:
            self.parts.append('\n')

    def handle_data(self, data):
        if not self._skip:
            self.parts.append(data)


def html_to_text(value) -> str:
    text = '' if value is None else str(value)
    if not _HAS_TAG.search(text):
        return _LEFTOVER_TAG.sub(' ', html.unescape(html.unescape(text)))
    parser = _TextExtractor()
    parser.feed(re.sub(r'[\r\n]+', ' ', text))
    parser.close()
    return _LEFTOVER_TAG.sub(' ', html.unescape(''.join(parser.parts)))


def normalise_ws(text: str) -> str:
    text = unicodedata.normalize('NFKC', str(text).translate(_ZERO_WIDTH))
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = '\n'.join(re.sub(r'[ \t\f\v]+', ' ', ln).strip() for ln in text.split('\n'))
    return re.sub(r'\n{3,}', '\n\n', text).strip()


_QUOTE_START = re.compile(
    r'(?im)^[ \t>]*(?:'
    r'-{2,}\s*(?:original message|original|原始邮件|forwarded message)\s*-{2,}'
    r'|原始邮件[ \t]*$'
    r'|(?:from|de|von|da|od|van|发件人)[ \t]*[:：][^\n]+\n(?:[^\n]*\n){0,4}?[ \t>]*'
    r'(?:sent|date|enviado|envoyé|gesendet|inviato|data|verzonden|wysłano|发送时间|发件时间|日期)[ \t]*[:：]'
    r'|(?:on|em|le|am|el|il|op)[ \t][^\n]{5,300}?(?:\n[^\n]{0,200}?)?[ \t](?:wrote|escreveu|a écrit|schrieb|escribió|ha scritto|schreef|napisał)[ \t]*:'
    r'|\d{4}年\d{1,2}月\d{1,2}日[^\n]{0,200}写道[ \t]*[:：]'
    r')')


def strip_quoted_history(text: str) -> tuple:
    """(text before the first reply header, chars removed). A pure quote is kept whole."""
    m = _QUOTE_START.search(text)
    if not m:
        return text, 0
    head = text[:m.start()].rstrip()
    if not head:
        return text, 0
    return head, len(text) - len(head)


_BOILERPLATE = [
    re.compile(r'(?is)⚠?[ \t]*external email\s*if the sender is unfamiliar,?\s*check\s*'
               r'home\.relx\.com\s*→?\s*trusted partners to verify them\.?'),
    re.compile(r'(?im)^[ \t]*caution:[ \t]*external (?:email|message)\b.*$'),
    re.compile(r'(?im)^[ \t]*visit[ \t]+elsevier support center[ \t]*$'),
]


def strip_boilerplate(text: str) -> str:
    for rx in _BOILERPLATE:
        text = rx.sub('', text)
    return text


def clean_message(value, keep_quoted: bool = False) -> tuple:
    """Raw VERBATIM -> (clean text, quoted-history chars removed)."""
    text = normalise_ws(html_to_text(value))
    cut = 0
    if not keep_quoted:
        text, cut = strip_quoted_history(text)
    return normalise_ws(strip_boilerplate(text)), cut


REPEAT_MIN = 80   # an earlier message this long, pasted verbatim into a later one, is removed from it


def strip_repeats(texts: list) -> tuple:
    """Remove earlier messages of the same ticket pasted back without a reply header."""
    out, cut = [], []
    for i, text in enumerate(texts):
        removed = 0
        for prev in texts[:i]:
            if len(prev) >= REPEAT_MIN and prev in text and text != prev:
                shorter = normalise_ws(text.replace(prev, ''))
                if shorter:
                    removed += len(text) - len(shorter)
                    text = shorter
        out.append(text)
        cut.append(removed)
    return out, cut


print('Cleaning functions defined.')

### Self-test — every rule, on the shapes seen in this export

In [ ]:
_CASES = [
    # (input, keep_quoted, expected clean text)
    ('<p>Hi</p><p>there&nbsp;you</p>', False, 'Hi\n\nthere you'),
    ('<div style="x">a<br />b</div><style>p{color:red}</style>', False, 'a\nb'),
    ('line one\nline   two', False, 'line one\nline two'),                     # plain text keeps newlines
    ('<p>see &lt;img src="x.png"&gt; here</p>', False, 'see here'),              # escaped twice
    ('mail me &lt;user3@example.com&gt;', False, 'mail me <user3@example.com>'),  # address survives
    ('BulkIDs\u200b\u00a0ok', False, 'BulkIDs ok'),
    ('<p>a&amp;nbsp;b</p>', False, 'a b'),                                    # escaped twice
    ('Thanks.\n\nFrom: NL Info <user2@example.com>\nSent: 18 May 2026\nTo: me\n\nold', False, 'Thanks.'),
    ('Thanks.\n\nFrom: NL Info <user2@example.com>\nSent: 18 May 2026\n\nold', True,
     'Thanks.\n\nFrom: NL Info <user2@example.com>\nSent: 18 May 2026\n\nold'),   # opening message
    ('Yes.\nOn Mon, 3 Aug 2026 at 10:00, NL Info <user2@example.com> wrote:\n> old', False, 'Yes.'),
    ('Ok\nEm qui., 13 de ago. de 2026 às 08:48, NL Info <user2@example.com> escreveu:\nold', False, 'Ok'),
    ('录屏\n\n-----原始邮件-----\n发件人:"NL" <user4@example.com>\n发送时间:2026-07-10', False, '录屏'),
    ('谢谢\n\n原始邮件\n\n发件人:x <user5@example.com>\n\n发件时间:2026年7月30日', False, '谢谢'),
    ('可以\n\n2026年7月14日 17:05,Helpdesk <user6@example.com> 写道:\n\nold', False, '可以'),
    ('From: a <user7@example.com>\nSent: today\n\nonly a quote', False,
     'From: a <user7@example.com>\nSent: today\n\nonly a quote'),                          # pure quote kept
    ('Hi\n\nCAUTION: External Message. Use caution opening links and attachments.', False, 'Hi'),
    ('Hi\n\nCAUTION: External email. Do not click links unless you know the sender.', False, 'Hi'),
    ('⚠ External email If the sender is unfamiliar, check home.relx.com → Trusted partners to '
     'verify them.\nBody', False, 'Body'),
    ('Kind regards,\n\nGiselle Tato\n\nVisit Elsevier Support Center', False, 'Kind regards,\n\nGiselle Tato'),
    # must NOT be cut: an address or a "from" that is not a reply header
    ('My email is from: my old job\nplease update it', False, 'My email is from: my old job\nplease update it'),
]
for _raw, _keep, _want in _CASES:
    _got = clean_message(_raw, keep_quoted=_keep)[0]
    assert _got == _want, f'\n  input : {_raw!r}\n  got   : {_got!r}\n  wanted: {_want!r}'
print(f'clean_message: all {len(_CASES)} cases pass.')

_long = 'Dear Wendy, we are also having additional issues with users not being able to access.' * 2
assert strip_repeats([_long, 'Please call me today.\n\n' + _long]) == \
    ([_long, 'Please call me today.'], [0, len(_long) + 2])
assert strip_repeats([_long, _long]) == ([_long, _long], [0, 0])          # an exact resend is kept
assert strip_repeats(['short', 'short and more']) == (['short', 'short and more'], [0, 0])
print('strip_repeats: all 3 cases pass.')

In [ ]:
msgs = msgs.sort_values(['INCIDENT_ID', 'ts', 'THREAD_ID']).reset_index(drop=True)
msgs['is_first'] = ~msgs['INCIDENT_ID'].duplicated()

_t0 = time.time()
_clean = [clean_message(v, keep_quoted=f) for v, f in zip(msgs['VERBATIM'], msgs['is_first'])]
msgs['msg_text'] = [t for t, _ in _clean]
msgs['quoted_chars_cut'] = [c for _, c in _clean]
# msgs is sorted by ticket, so groups come back in row order.
_rep = msgs.groupby('INCIDENT_ID', sort=False)['msg_text'].apply(lambda s: strip_repeats(s.tolist()))
msgs['msg_text'] = [t for texts, _ in _rep for t in texts]
msgs['repeat_chars_cut'] = [c for _, cuts in _rep for c in cuts]
msgs['quoted_chars_cut'] += msgs['repeat_chars_cut']
print(f'Cleaned {len(msgs):,} messages in {time.time() - _t0:.1f}s')
print(f'Earlier messages pasted back without a header, removed from '
      f'{int((msgs["repeat_chars_cut"] > 0).sum())} replies ({int(msgs["repeat_chars_cut"].sum()):,} chars)')

_raw_chars, _clean_chars = msgs['VERBATIM'].str.len().sum(), msgs['msg_text'].str.len().sum()
print(f'Characters: {_raw_chars:,} raw -> {_clean_chars:,} clean ({_clean_chars / _raw_chars:.0%})')
print(f'Quoted history cut from {int((msgs["quoted_chars_cut"] > 0).sum()):,} replies '
      f'({int(msgs["quoted_chars_cut"].sum()):,} chars)')
print(f'Opening messages kept whole (may contain a forwarded request): '
      f'{int((msgs["is_first"] & msgs["msg_text"].str.contains(_QUOTE_START)).sum())}')

_tag_left = msgs['msg_text'].str.contains(r'<(?![^<>]*@)\s*/?\s*(?!inf\b)[a-z][a-z0-9:]*(?=[\s/>])[^<>]*>', case=False)
_ent_left = msgs['msg_text'].str.contains(r'&(?:[a-z]+|#\d+);', case=False)
assert not _tag_left.any(), msgs.loc[_tag_left, 'msg_text'].head(3).tolist()
assert not _ent_left.any(), msgs.loc[_ent_left, 'msg_text'].head(3).tolist()
print('No HTML tags or entities left (Embase record markup such as <inf> is content and kept).')

EMPTY_MSG = '[no text - attachment or image only]'
_empty = msgs['msg_text'] == ''
msgs.loc[_empty, 'msg_text'] = EMPTY_MSG
print(f'Messages with no text after cleaning: {int(_empty.sum())} (kept, marked {EMPTY_MSG!r})')

## 3. Rebuild one conversation per ticket

Every message becomes `[YYYY-MM-DD HH:MM] SPEAKER: text`, in `DATE_CREATED` order, under a
`Channel:` header. `customer_text` is the customer's side alone (`CUSTOMER` and
`CUSTOMER (logged by agent)`), used by the keyword audit in section 4.

Conversations over 16,000 characters are sent to the model as head 4,000 + tail 11,900 with a visible
marker; the `Conversation` column keeps the full text.

In [ ]:
msgs['speaker'] = msgs['THREAD_ENTRY_TYPE'].map(SPEAKER)
msgs['line'] = ('[' + msgs['ts'].dt.strftime('%Y-%m-%d %H:%M') + '] '
                + msgs['speaker'] + ': ' + msgs['msg_text'])
_is_cust = msgs['speaker'].str.startswith('CUSTOMER')

tickets = msgs.groupby('INCIDENT_ID', sort=False).agg(
    INCIDENT_REF=('INCIDENT_REF', 'first'),
    channel=('INCIDENT_CHANNEL', lambda s: CHANNEL_NAME[s.iloc[0]]),
    incident_type=('INCIDENT_TYPE', 'first'),
    contact_id=('CONTACT_ID', 'first'),
    first_message=('ts', 'min'),
    last_message=('ts', 'max'),
    n_messages=('THREAD_ID', 'size'),
    n_customer_msgs=('speaker', lambda s: int(s.eq('CUSTOMER').sum())),
    n_logged_by_agent=('speaker', lambda s: int(s.eq('CUSTOMER (logged by agent)').sum())),
    n_agent_msgs=('speaker', lambda s: int(s.eq('AGENT').sum())),
    quoted_chars_cut=('quoted_chars_cut', 'sum'),
).reset_index()

_body = msgs.groupby('INCIDENT_ID', sort=False)['line'].agg('\n\n'.join)
_cust_text = msgs[_is_cust].groupby('INCIDENT_ID', sort=False)['msg_text'].agg('\n\n'.join)
_first_cust = msgs[_is_cust].groupby('INCIDENT_ID', sort=False)['msg_text'].first()
tickets[TEXT_COL] = ('Channel: ' + tickets['channel'] + '\n\n'
                     + tickets['INCIDENT_ID'].map(_body))
tickets['customer_text'] = tickets['INCIDENT_ID'].map(_cust_text).fillna('')
tickets['first_customer_msg'] = tickets['INCIDENT_ID'].map(_first_cust).fillna('')
tickets['month'] = tickets['first_message'].dt.to_period('M').astype(str)

TRUNC_MARKER = '\n\n...[MIDDLE OF CONVERSATION OMITTED FOR LENGTH]...\n\n'
assert TRUNC_HEAD + len(TRUNC_MARKER) + TRUNC_TAIL <= TRUNC_CAP


def cap(text: str) -> str:
    text = str(text)
    return text if len(text) <= TRUNC_CAP else text[:TRUNC_HEAD] + TRUNC_MARKER + text[-TRUNC_TAIL:]


tickets['conversation_chars'] = tickets[TEXT_COL].str.len()
tickets['conversation_sent'] = tickets[TEXT_COL].map(cap)
tickets['was_truncated'] = tickets['conversation_chars'] > TRUNC_CAP
tickets[ID_COL] = tickets[ID_COL].astype(str).str.strip()

assert tickets[ID_COL].is_unique and len(tickets) == raw['INCIDENT_ID'].nunique()
assert tickets['n_messages'].sum() == len(msgs), 'a message was lost in the rebuild'
assert tickets['conversation_sent'].str.len().max() <= TRUNC_CAP

_no_cust = tickets['customer_text'] == ''
print(f'{len(tickets):,} conversations rebuilt from {len(msgs):,} messages')
print(f'  messages per ticket: median {tickets["n_messages"].median():.0f}, max {tickets["n_messages"].max()}')
print(f'  no customer turn at all (agent-only notes): {int(_no_cust.sum())}')
print(f'  truncated to {TRUNC_CAP:,} chars for the model: {int(tickets["was_truncated"].sum())}')
print('\nTickets per channel:')
print(tickets['channel'].value_counts().to_string())

_CONV_COLS = [ID_COL, 'INCIDENT_ID', 'channel', 'incident_type', 'contact_id', 'first_message',
              'last_message', 'n_messages', 'n_customer_msgs', 'n_logged_by_agent', 'n_agent_msgs',
              'quoted_chars_cut', 'conversation_chars', 'was_truncated', TEXT_COL]
tickets[_CONV_COLS].to_csv(CONV_CSV, index=False, encoding='utf-8-sig')
print(f'\nWrote {CONV_CSV.relative_to(BASE)}')

### Read one conversation per channel

In [ ]:
for _ch in ['Web', 'Email', 'Chat', 'Phone', 'Fax']:
    _pool = tickets[(tickets['channel'] == _ch) & (tickets['n_messages'] >= 3)]
    _pool = _pool if len(_pool) else tickets[tickets['channel'] == _ch]
    _r = _pool.sample(1, random_state=1).iloc[0]
    print('=' * 100)
    print(f'{_ch}  |  {_r[ID_COL]}  |  {_r["n_messages"]} messages '
          f'({_r["n_customer_msgs"]} customer, {_r["n_logged_by_agent"]} logged by agent, '
          f'{_r["n_agent_msgs"]} agent)')
    print('=' * 100)
    _t = _r[TEXT_COL]
    print(_t if len(_t) <= 3500 else _t[:3500] + '\n...[cut for display]')
    print()

## 4. Keyword audit column *(free — a cross-check, not the tag)*

A regex over the **customer side only**. It is not used to tag anything; section 7 crosstabs it
against the LLM gate so the disagreements can be read by hand, which is the quickest way to spot a
definition that is too wide or too narrow.

In [ ]:
EMAILREG_KW = re.compile(
    r'regist|sign[\s-]?up|verif\w*\s+(?:e-?mail|link|code)|activat|confirm\w*\s+(?:e-?mail|link|account)'
    r'|(?:change|update|wrong|different|correct|institutional|personal|work|university|old|new)\s+'
    r'(?:my\s+)?e-?mail|e-?mail\s+(?:address\s+)?(?:is\s+)?(?:already|not\s+(?:recogni|found|valid|exist))'
    r'|already\s+(?:registered|in\s+use|exists)', re.I)

tickets['kw_email_reg'] = tickets['customer_text'].map(lambda t: bool(EMAILREG_KW.search(t)))
tickets['kw_term'] = tickets['customer_text'].map(
    lambda t: (m.group(0).lower() if (m := EMAILREG_KW.search(t)) else ''))
print(f'Keyword hit on the customer side: {int(tickets["kw_email_reg"].sum())} of {len(tickets)}')
print(tickets.groupby('channel')['kw_email_reg'].agg(['sum', 'size']).to_string())
print('\nMost common matched terms:')
print(tickets.loc[tickets['kw_email_reg'], 'kw_term'].value_counts().head(15).to_string())

## 5. Pipeline definition workbook, build, and gate 1 (prompt review)

The workbook holds the gate from the start. The two discovered modules are added in section 8:
first with a one-leaf placeholder taxonomy and `discovery_mode='full'` (discover reads its prompt
and prefix from the built config), then with the curated taxonomy. `write_definition()` rewrites all
three sheets each time, so the gate rows are always identical.

In [ ]:
# ---- Shared prompt language ---------------------------------------------
G_THREAD = (
    'Read the whole conversation before deciding. Each message is labelled CUSTOMER, AGENT, or '
    'CUSTOMER (logged by agent) - the last is a note an agent typed on behalf of the customer, '
    'such as the subject of a phone call, and counts as the customer speaking. The request is often '
    'restated or corrected further down the thread.')
G_WHO = (
    'Decide on what the CUSTOMER is asking for or reporting, not on what the agent writes. Agents '
    'routinely tell customers to sign in or register with their email address, ask which email '
    'address was used, or quote the email address on file. None of that makes a ticket about email '
    'registration unless the problem the customer raises is the email address or the account '
    'registration itself.')
G_FORWARD = (
    'The person writing as CUSTOMER is sometimes an Elsevier colleague, or a librarian or '
    'administrator, writing on behalf of users. Judge the request they are passing on.')
G_ONE = 'Exactly one code. If two seem to fit, pick the one the customer would name themselves.'

# ---- Stage A: the gate --------------------------------------------------
G_YES = 'Email registration / email-account issue'
G_NO  = 'Not email registration'
GATE_ORDER = [G_YES, G_NO]
GATE_DEFS = {
    G_YES: (
        'The problem or request is about the email address tied to an Elsevier or Embase user '
        'account, or about registering an account with an email address. This includes: creating '
        'or registering an Embase or Elsevier account; a verification, activation or confirmation '
        'email that did not arrive or whose link does not work; registering or activating remote '
        'access with an institutional email address; an email address reported as already '
        'registered, not recognised or not found; which email to use - institutional versus '
        'personal - and moving an account or its access from one email address to another; '
        'changing, correcting, merging or removing the email address on an account; and being '
        'unable to sign in or reset a password because the registered email address is wrong, '
        'unreachable or unknown to the customer. It also includes an organisation asking for user '
        'accounts, registration IDs or passcodes to be created for named people, or for users to '
        'be added, removed or deactivated, because each of those registers or de-registers a user '
        'account.'),
    G_NO: (
        'Everything else: content and indexing questions or corrections, missing or wrong records, '
        'coverage, search and product features, error messages, exports, alerts that are not about '
        'the registered email, linking, institutional access set-up by IP or SSO where no user '
        'email is in question, orders, renewals and invoices, usage reports, training and '
        'feedback. A ticket is not about email registration just because an email address appears '
        'in it, because the agent asks which email address was used, or because the customer is '
        'told to sign in with their email address. Changing the email address of an AUTHOR on a '
        'published paper or record is a content correction, not an account issue. Bounce-back and '
        'out-of-office messages are not evidence either way.'),
}
MOD_DESC_G = (
    'Decide whether this Embase support ticket is about email registration or an email-account '
    f'issue. {G_THREAD} {G_WHO} {G_FORWARD} {G_ONE} '
    f'Assign the single best-fitting LEAF code (e.g. {PREFIX_G}1.1). Never return an empty code - '
    f'if the ticket does not show an email-account problem, for any reason including that it shows '
    f'nothing at all, the answer is {G_NO}.')


def _fields_for(module, stub, prefix, what, alternative_phrase):
    return [
        {'module_name': module, 'field_name': f'{stub}_code', 'field_type': 'code',
         'description': 'Assigned leaf code',
         'prompt_instruction': (
             f'Assign the single leaf-level code (e.g. {prefix}1.1) for the {what} that best '
             'matches this ticket. Read each Definition in full and match on what the ticket '
             'shows, not on surface keywords. Only use codes from the taxonomy.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'TRUE', 'notes': None},
        {'module_name': module, 'field_name': f'{stub}_justification', 'field_type': 'str',
         'description': 'Why this code was chosen over the nearest alternative',
         'prompt_instruction': (
             f'In one or two sentences, explain why this {what} applies and why '
             f'{alternative_phrase} does not. Refer to what the customer actually wrote. Do not '
             'use apostrophes or quotation marks in your answer.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
        {'module_name': module, 'field_name': f'{stub}_evidence', 'field_type': 'str',
         'description': 'Verbatim supporting quote from the transcript',
         'prompt_instruction': (
             'Copy verbatim the single sentence or phrase from the transcript that most strongly '
             'supports this choice, preferring the words of the customer. Do not paraphrase, '
             'infer, or add words that are not in the transcript. If no direct evidence exists, '
             'return an empty string.'),
         'in_output': 'TRUE', 'explode_list': 'FALSE', 'validate_enum': 'FALSE', 'notes': None},
    ]


_COLS = {
    'modules': ['module_name', 'taxonomy_prefix', 'taxonomy_file', 'reorder_group',
                'reorder_order', 'enabled', 'description', 'notes', 'discovery_mode',
                'extract_prompt'],
    'fields': ['module_name', 'field_name', 'field_type', 'description', 'prompt_instruction',
               'in_output', 'explode_list', 'validate_enum', 'notes'],
    'taxonomy': ['module_name', 'level', 'label', 'description', 'notes'],
}


def module_spec(module, *, order, description, taxonomy, notes, what, alt,
                discovery_mode='', extract_prompt=''):
    """One extra module for write_definition(). taxonomy: [(label, definition), ...]."""
    return {'module': module, 'what': what, 'alt': alt, 'taxonomy': taxonomy, 'notes': notes,
            'row': {'module_name': module, 'taxonomy_prefix': MODULE_PREFIXES[module],
                    'taxonomy_file': f'{module}.json', 'reorder_group': module, 'reorder_order': order,
                    'enabled': 'TRUE', 'description': description, 'notes': notes,
                    'discovery_mode': discovery_mode, 'extract_prompt': extract_prompt}}


STUB_OF = {MOD_G: STUB_G, MOD_S: STUB_S, MOD_C: STUB_C}


def write_definition(extra=()) -> list:
    """Write DEFN_XLSX with the gate plus any module_spec() entries; return the module names."""
    modules_rows = [
        {'module_name': MOD_G, 'taxonomy_prefix': PREFIX_G, 'taxonomy_file': f'{MOD_G}.json',
         'reorder_group': 'G', 'reorder_order': 0, 'enabled': 'TRUE', 'description': MOD_DESC_G,
         'notes': 'Stage A. Runs on every ticket. Authored for this notebook.'}]
    fields_rows = _fields_for(MOD_G, STUB_G, PREFIX_G, 'verdict', 'the other verdict')
    taxonomy_rows = []
    for _label in GATE_ORDER:
        taxonomy_rows += [
            {'module_name': MOD_G, 'level': 'L1', 'label': _label},
            {'module_name': MOD_G, 'level': 'L2', 'label': _label,
             'description': f'Definition: {GATE_DEFS[_label]}', 'notes': 'Authored for this run.'}]
    for spec in extra:
        m = spec['module']
        modules_rows.append(spec['row'])
        fields_rows += _fields_for(m, STUB_OF[m], MODULE_PREFIXES[m], spec['what'], spec['alt'])
        for _label, _defn in spec['taxonomy']:
            taxonomy_rows += [
                {'module_name': m, 'level': 'L1', 'label': _label},
                {'module_name': m, 'level': 'L2', 'label': _label,
                 'description': f'Definition: {_defn}', 'notes': spec['notes']}]

    frames = {'modules': pd.DataFrame(modules_rows).reindex(columns=_COLS['modules']),
              'fields': pd.DataFrame(fields_rows).reindex(columns=_COLS['fields']),
              'taxonomy': pd.DataFrame(taxonomy_rows).reindex(columns=_COLS['taxonomy'])}
    with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as xw:
        for sheet, df in frames.items():
            df.to_excel(xw, sheet_name=sheet, index=False)

    back = pd.read_excel(DEFN_XLSX, sheet_name=None, dtype=str)
    for sheet, df in frames.items():
        assert len(back[sheet]) == len(df), f'{sheet}: row count changed on write'
    tb = back['taxonomy']
    for spec_mod, pairs in [(MOD_G, GATE_DEFS.items())] + [(s['module'], s['taxonomy']) for s in extra]:
        for _label, _defn in pairs:
            _row = tb[(tb.module_name == spec_mod) & (tb.level == 'L2') & (tb.label == _label)]
            assert len(_row) == 1 and _defn in _row.iloc[0]['description'], \
                f'{spec_mod}: row lost or altered on write: {_label}'
    mods = back['modules']['module_name'].tolist()
    print(f'Wrote {DEFN_XLSX.relative_to(BASE)}: modules {mods}')
    for sheet, df in frames.items():
        print(f'  {sheet:<9} {len(df):>3} rows')
    return mods


BUILT_MODULES = write_definition()

In [ ]:
def build(modules: list) -> dict:
    run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
             '--input', str(DEFN_XLSX), '--force', *modules], 'build_pipeline')
    cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
    assert set(cfg['modules']) == set(modules), f'config holds {sorted(cfg["modules"])}'
    for m in modules:
        assert (TAXONOMY_DIR / f'{m}.json').exists(), f'missing taxonomy for {m}'
        assert (MODULES_DIR / f'{m}.py').exists(), f'missing {m}.py'
        assert cfg['modules'][m]['taxonomy_prefix'] == MODULE_PREFIXES[m]
    return cfg


cfg = build(BUILT_MODULES)

# Output columns must not collide with the prefixes drop_enrichment_cols() strips.
from rca_pipeline.utils import _STRUCTURAL_PREFIXES
OWN_COLS = [f'{s}_' for s in STUB_OF.values()] + [
    'is_email_registration', 'gate', 'email_reg_subcategory', 'main_contact_reason', 'final_tag',
    'kw_email_reg', 'kw_term', 'channel', 'customer_text', 'first_customer_msg',
    'conversation_chars', 'conversation_sent', 'was_truncated', 'quoted_chars_cut',
    *META_COLS.values()]
for _c in OWN_COLS:
    for _bad in _STRUCTURAL_PREFIXES:
        assert not _c.startswith(_bad), f'column {_c!r} collides with structural prefix {_bad!r}'
for _a in MODULE_PREFIXES.values():
    for _b in MODULE_PREFIXES.values():
        assert _a == _b or not _a.startswith(_b), f'{_a} and {_b} overlap'

GATE_LABELS = leaf_labels(MOD_G)
CODE_OF_GATE = {v: k for k, v in GATE_LABELS.items()}
assert set(GATE_LABELS.values()) == set(GATE_ORDER)
print(f'\n{MOD_G}:')
for _c, _l in GATE_LABELS.items():
    print(f'  {_c:<8} {_l}')

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

PROMPT_FINGERPRINTS = {}
prompts = {}


def render_prompts(modules: list) -> None:
    """Render each module's system prompt as the model sees it, and fingerprint it."""
    importlib.invalidate_caches()
    tuning_sig = '|'.join(f'{k}={TUNING[k]}' for k in OUTPUT_AFFECTING)
    preproc = f'{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}'
    for module in modules:
        tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
        name = f'generated.modules.{module}'
        mod = importlib.reload(sys.modules[name]) if name in sys.modules \
            else importlib.import_module(name)
        prompts[module] = mod._build_system_prompt(
            taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))
        PROMPT_FINGERPRINTS[module] = hashlib.sha256(
            (prompts[module] + tuning_sig + preproc).encode('utf-8')).hexdigest()[:16]
    print(f'{"module":<26} {"chars":>7} {"~tokens":>8}  fingerprint')
    for module in modules:
        print(f'{module:<26} {len(prompts[module]):>7,} {len(prompts[module]) // 4:>8,}  '
              f'{PROMPT_FINGERPRINTS[module]}')


render_prompts(BUILT_MODULES)

# ---- Verbatim check -----------------------------------------------------
_checks = [(d, MOD_G, f'definition of {l}') for l, d in GATE_DEFS.items()]
_checks += [(l, MOD_G, 'label') for l in GATE_ORDER]
_checks += [(t, MOD_G, n) for t, n in ((G_THREAD, 'thread rule'), (G_WHO, 'customer-not-agent rule'),
                                       (G_FORWARD, 'forwarding rule'))]
for _text, _module, _what in _checks:
    assert _text in prompts[_module], f'{_what} missing from the {_module} prompt'
print(f'\nVerbatim check: all {len(_checks)} strings present in the prompts as written.')

In [ ]:
print('=' * 78)
print(f'FULL GATE SYSTEM PROMPT  ({MOD_G})')
print('=' * 78)
print(prompts[MOD_G])

## 6. Profiles, stage runner, checkpoint guard

`rca run` silently ignores a profile key it does not recognise, so every key is checked
against `--help` before it is written. The checkpoint guard discards a stage's checkpoint when the
prompt fingerprint has changed — otherwise a re-run after editing a definition would replay the old
answers (`skip=N todo=0`) without making a single call.

In [ ]:
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests


def write_profile(path: Path, *, comment, input_path, output_path, module) -> Path:
    spec = {'_comment': comment,
            'input': input_path.relative_to(BASE).as_posix(),
            'output': output_path.relative_to(BASE).as_posix(),
            'taxonomy_dir': 'generated/taxonomies',
            'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': [module], **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


_NB = 'notebooks/19_embase_2026_email_registration.ipynb'
write_profile(PROFILE_G, module=MOD_G, input_path=INPUT_G, output_path=OUT_G, comment=(
    'Stage A: is this Embase ticket about email registration or an email-account issue? Input is '
    'every ticket of the 2026-07-27 Embase thread export, cleaned and rebuilt as one '
    f'speaker-labelled conversation per ticket by {_NB}.'))
write_profile(PROFILE_S, module=MOD_S, input_path=INPUT_S, output_path=OUT_S, comment=(
    'Stage B: sub-category of email-account issue, stage-A positives only. Taxonomy discovered '
    f'from those tickets and curated in {_NB} section 8.'))
write_profile(PROFILE_C, module=MOD_C, input_path=INPUT_C, output_path=OUT_C, comment=(
    'Stage C: main contact reason, stage-A negatives only. Embase contact reason taxonomy '
    f'discovered from those conversations and curated in {_NB} section 8.'))
for _p in (PROFILE_G, PROFILE_S, PROFILE_C):
    print(f'wrote {_p.relative_to(BASE)}')
print(PROFILE_G.read_text(encoding='utf-8'))

In [ ]:
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list:
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl', CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, module: str) -> None:
    """Drop this input's checkpoint if it was produced by a different prompt."""
    want = PROMPT_FINGERPRINTS.get(module)
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    if want is None or stored.get(module) == want:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  {module}: prompt changed since that checkpoint '
              f'({stored.get(module) or "no fingerprint"} -> {want}); discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, module: str) -> None:
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    stored[module] = PROMPT_FINGERPRINTS[module]
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, module: str, input_path=None, output_path=None,
              label: str = '', fresh: bool = False) -> Path:
    """Run one stage as a subprocess. fresh=True forces real calls (smoke tests only)."""
    spec = json.loads(profile.read_text(encoding='utf-8'))
    in_path = Path(input_path) if input_path else BASE / spec['input']
    out_path = Path(output_path) if output_path else BASE / spec['output']
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if fresh:
        for p in checkpoint_files(in_path) + [_fingerprint_file(in_path)]:
            p.unlink(missing_ok=True)
    else:
        _sync_checkpoint(in_path, module)
    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(in_path), '--output', str(out_path), '--modules', module]
    print(f'\n{"=" * 78}\n{label or module}\n{"=" * 78}\n$ {" ".join(cmd)}\n')
    _t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            encoding='utf-8', errors='replace', bufsize=1, env=SUBPROCESS_ENV)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'{label or module} failed (exit {rc})')
    assert out_path.exists(), f'{label or module} reported success but wrote no output'
    _stamp_checkpoint(in_path, module)
    print(f'\n{label or module}: {time.time() - _t0:.0f}s -> {out_path.relative_to(BASE)}')
    return out_path


def read_labels(path, stub: str, code_to_label: dict, out_col: str) -> pd.DataFrame:
    """Stage output -> id, code, label, justification, evidence, grounded. Unknown codes -> ''."""
    df = read_stage_output(path)
    code = df[f'{stub}_code'].fillna('').astype(str).str.strip()
    out = pd.DataFrame({
        ID_COL: df[ID_COL],
        f'{out_col}_code': code,
        out_col: code.map(code_to_label).fillna(''),
        f'{out_col}_justification': df.get(f'{stub}_justification', pd.Series('', index=df.index)).fillna(''),
        f'{out_col}_evidence': df.get(f'{stub}_evidence', pd.Series('', index=df.index)).fillna(''),
    })
    if f'{stub}_evidence_grounded' in df.columns:
        out[f'{out_col}_evidence_grounded'] = df[f'{stub}_evidence_grounded']
    return out


def write_input(frame: pd.DataFrame, path: Path) -> Path:
    frame[[ID_COL, 'conversation_sent']].rename(columns={'conversation_sent': TEXT_COL}) \
        .to_csv(path, index=False, encoding='utf-8')
    return path


print('Stage runner and checkpoint guard defined.')

## 7. Stage A — the email-registration gate

### Gate 2: smoke test *(cents)*

20 tickets stratified over channel (the one Fax ticket included), real calls (`fresh=True`). Read the justifications: a YES should
point at an email-account problem **the customer raised**; a NO should name what the customer
actually wanted.

In [ ]:
SMOKE_N, SMOKE_SEED = 20, 0
_parts = [g.sample(n=min(4, len(g)), random_state=SMOKE_SEED)
          for _, g in tickets.groupby('channel')]
_seeded = pd.concat(_parts)
_rest = tickets[~tickets[ID_COL].isin(_seeded[ID_COL])].sample(
    n=max(0, SMOKE_N - len(_seeded)), random_state=SMOKE_SEED)
smoke = pd.concat([_seeded, _rest]).head(SMOKE_N).reset_index(drop=True)
print(smoke['channel'].value_counts().to_string())

smoke_g = read_labels(
    run_stage(PROFILE_G, module=MOD_G, input_path=write_input(smoke, SMOKE_DIR / 'smoke_gate_input.csv'),
              output_path=SMOKE_DIR / 'smoke_gate.xlsx', label=f'SMOKE {MOD_G}', fresh=True),
    STUB_G, GATE_LABELS, 'gate')
assert set(smoke_g[ID_COL]) == set(smoke[ID_COL])
assert (smoke_g['gate'] != '').all(), smoke_g[smoke_g['gate'] == ''].to_string()

_j = smoke.merge(smoke_g, on=ID_COL)
print('\n' + _j['gate'].value_counts().to_string())
for _, _r in _j.sort_values('gate').iterrows():
    print(f'\n[{_r[ID_COL]}] {_r["channel"]:<5}  kw={"Y" if _r["kw_email_reg"] else "-"}  -> {_r["gate"]}')
    print(f'   {_r["gate_justification"]}')
    print(f'   evidence: {str(_r["gate_evidence"])[:200]}')

### Decision point

Continue when the YES verdicts name an email-account problem the customer raised and the NO verdicts
name what the customer actually wanted. To reword a definition, edit `GATE_DEFS` in section 5 and
re-run from there; the fingerprint guard discards the stale checkpoint.

In [ ]:
STOP_AFTER_SMOKE = False   # set True to pause here and read the smoke output first
if STOP_AFTER_SMOKE:
    raise SystemExit('STOP_AFTER_SMOKE is True - read the smoke output, then set it to False.')
print('Continuing to the full gate run.')

### Full gate run — every ticket

In [ ]:
write_input(tickets, INPUT_G)
gate = read_labels(run_stage(PROFILE_G, module=MOD_G, label=f'FULL {MOD_G} - {len(tickets)} tickets'),
                   STUB_G, GATE_LABELS, 'gate')
assert set(gate[ID_COL]) == set(tickets[ID_COL]), 'the gate returned a different id set'
assert (gate['gate'] != '').all(), f'{int((gate["gate"] == "").sum())} blank or off-taxonomy codes'
tickets = tickets.drop(columns=[c for c in gate.columns if c != ID_COL and c in tickets.columns]) \
                 .merge(gate, on=ID_COL, how='left', validate='1:1')
tickets['is_email_registration'] = tickets['gate'].eq(G_YES)

_n_yes = int(tickets['is_email_registration'].sum())
print(f'\nEmail registration / email-account issue: {_n_yes} of {len(tickets)} '
      f'({_n_yes / len(tickets):.1%})')
print(pd.crosstab(tickets['channel'], tickets['is_email_registration'], margins=True).to_string())
if 'gate_evidence_grounded' in tickets.columns:
    print(f'\nEvidence quotes found in the transcript: {tickets["gate_evidence_grounded"].mean():.1%}')

In [ ]:
print('LLM gate vs keyword audit (customer side):')
print(pd.crosstab(tickets['is_email_registration'].map({True: 'LLM yes', False: 'LLM no'}),
                  tickets['kw_email_reg'].map({True: 'kw hit', False: 'no kw'})).to_string())

for _title, _mask in (('Keyword hit, LLM says NO  (is the definition too narrow?)',
                       tickets['kw_email_reg'] & ~tickets['is_email_registration']),
                      ('LLM says YES, no keyword  (phrasings the regex has no pattern for)',
                       ~tickets['kw_email_reg'] & tickets['is_email_registration'])):
    _d = tickets[_mask]
    print(f'\n{"=" * 100}\n{_title}: {len(_d)}\n{"=" * 100}')
    for _, _r in _d.head(15).iterrows():
        print(f'\n[{_r[ID_COL]}] {_r["channel"]}  kw={_r["kw_term"]!r}')
        print(f'   customer: {_r["first_customer_msg"][:250]!r}')
        print(f'   -> {_r["gate"]}: {_r["gate_justification"]}')

## 8. Derive the two taxonomies — stage B sub-categories and the stage C contact reasons

Both are drafted by `rca build discover`, over **every** ticket in their population:

- **B** over the stage-A positives: what specifically about the email address or registration went
  wrong.
- **C** over the stage-A negatives: why the customer contacted Embase support.

Discover runs a per-ticket extraction (pass 1, whose item counts are printed below) and then
normalises the items into a draft taxonomy (pass 2). *Known quirk:* it can exit with `0xC0000005` at
interpreter shutdown after writing everything, so it is judged by the draft file's modification
time and the `DISCOVERY COMPLETE` log line, not by the exit code.

The drafts are then **curated by hand** into `SUBCATEGORIES` and `CONTACT_REASONS`, from:

- the draft;
- the pass-1 item counts;
- a read of the tickets;
- for C, the agent's own contact reasons.

Nothing is classified until both lists have been reviewed; the notebook stops before stage B.

In [ ]:
positives = tickets[tickets['is_email_registration']].copy()
negatives = tickets[~tickets['is_email_registration']].copy()
write_input(positives, INPUT_S)
write_input(negatives, INPUT_C)
print(f'{len(positives)} email-registration tickets -> {INPUT_S.relative_to(BASE)}')
print(f'{len(negatives)} other tickets              -> {INPUT_C.relative_to(BASE)}')

_JSON_TAIL = (
    '\n\nReturn a JSON array - one object per record:\n'
    '[{"uid": "<uid>", "extracted_items": [{"item": "<abstract category>", "count": 1}]}]\n'
    'Return ONLY valid JSON.')
EXTRACT_PROMPT_S = (
    'You are analysing Embase customer support tickets that are already known to be about an '
    'email-account or email-registration problem. For each transcript, identify the ABSTRACT '
    'sub-type of that problem - what specifically about the email address or the registration went '
    'wrong, or is being requested. Lines are labelled CUSTOMER or AGENT; describe the problem of '
    'the customer, not the fix the agent applied.\n\n'
    'CRITICAL RULES:\n'
    '- Merge similar items into ONE category and use generic names that would work as column '
    'headers in a report, for example "Verification email not received" or "Remote access '
    'registration with institutional email".\n'
    '- Each transcript should produce exactly 1 item: the main email-account problem.' + _JSON_TAIL)
EXTRACT_PROMPT_C = (
    'You are an expert at categorising customer support contacts for Embase, the biomedical and '
    'pharmacological literature database from Elsevier. For each transcript, identify the ABSTRACT '
    'contact reason - the high-level category of why the customer contacted support, NOT the '
    'surface wording and NOT what the agent did. Lines are labelled CUSTOMER or AGENT; the person '
    'writing as CUSTOMER may be a librarian, a publisher or an Elsevier colleague writing on '
    'behalf of others.\n\n'
    'CRITICAL RULES:\n'
    '- Merge similar reasons into ONE category. "Article missing from Embase", "journal not '
    'indexed", "records not loaded" are one reason if the customer wants content added.\n'
    '- Use GENERIC category names that would work as column headers in a report, for example '
    '"Missing or incomplete indexed content", "Correction to a record", "Search or export '
    'feature question", "Institutional access set-up", "Order, renewal or invoice".\n'
    '- Each transcript should produce exactly 1 item: the main reason for the contact.' + _JSON_TAIL)

PLACEHOLDER = [('Placeholder for discovery', 'Placeholder - replaced after discovery.')]
DISCOVER_SPECS = [
    module_spec(MOD_S, order=2, taxonomy=PLACEHOLDER, notes='Discovery placeholder.',
                what='sub-category', alt='the closest alternative sub-category',
                description='Sub-type of email-account problem. Placeholder taxonomy for discovery.',
                discovery_mode='full', extract_prompt=EXTRACT_PROMPT_S),
    module_spec(MOD_C, order=3, taxonomy=PLACEHOLDER, notes='Discovery placeholder.',
                what='contact reason', alt='the closest alternative contact reason',
                description='Main contact reason. Placeholder taxonomy for discovery.',
                discovery_mode='full', extract_prompt=EXTRACT_PROMPT_C),
]
build(write_definition(DISCOVER_SPECS))

In [ ]:
# The drafts only seed the hand-curated lists below, so a re-run reuses them rather than paying
# for discovery again. The curated lists, not the drafts, decide the codes.
REDISCOVER = False


def discover(module: str, frame: pd.DataFrame, input_path: Path) -> dict:
    """Run discover over every row of frame; print and return the draft taxonomy."""
    draft_path = DRAFT_DIR / f'{module}_draft.json'
    if draft_path.exists() and not REDISCOVER:
        print(f'{module}: reusing the existing draft (set REDISCOVER = True to redraw it)')
    else:
        _t0 = time.time()
        r = run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'discover',
                     '--input', str(DEFN_XLSX), '--data', str(input_path), '--modules', module,
                     '--sample-n', str(len(frame)), '--id-col', ID_COL, '--text-col', TEXT_COL,
                     '--batch-size', '5'],
                    f'discover {module}', ok_codes=(0, 3221225477, -1073741819))   # 0xC0000005
        assert draft_path.exists() and draft_path.stat().st_mtime >= _t0, \
            f'discover did not write a fresh {draft_path.name}'
        assert 'DISCOVERY COMPLETE' in (r.stdout + r.stderr), 'discover did not report completion'
    draft = json.loads(draft_path.read_text(encoding='utf-8'))
    print(f'\n{"=" * 100}\nDRAFT {module}: {draft_path.relative_to(BASE)}\n{"=" * 100}')
    for code_, node in draft.items():
        print(f'\n{code_}  {node.get("label")}\n    {node.get("description", "")}')
        for c2, leaf in node.get('children', {}).items():
            print(f'  {c2:<8} {leaf.get("label")}\n           {leaf.get("description", "")}')
    extracted = DRAFT_DIR / f'{module}_extracted.json'
    if extracted.exists():
        items = json.loads(extracted.read_text(encoding='utf-8')).get('items', [])
        print(f'\nPass-1 items (top 40 of {len(items)}):')
        for it in sorted(items, key=lambda x: -x.get('frequency', 0))[:40]:
            print(f'  {it.get("frequency", 0):>4}  {it.get("item")}')
    return draft


draft_s = discover(MOD_S, positives, INPUT_S)

In [ ]:
draft_c = discover(MOD_C, negatives, INPUT_C)

In [ ]:
# The deep-dive reading: the positives, what the customer raised, and why the gate said yes.
for _, _r in positives.sort_values('channel').iterrows():
    print(f'\n[{_r[ID_COL]}] {_r["channel"]}  ({_r["n_messages"]} msgs)')
    print(f'   customer : {_r["first_customer_msg"][:400]!r}')
    print(f'   evidence : {str(_r["gate_evidence"])[:200]!r}')
    print(f'   why      : {_r["gate_justification"]}')

In [ ]:
# The negatives, grouped by the agent's own contact reason, 4 examples each - so the curated
# contact reasons can be checked against what the tickets actually say.
_neg = negatives.merge(meta, on='INCIDENT_ID', how='left', validate='1:1')
for (_l1, _l2), _g in sorted(_neg.groupby(['agent_cr_l1', 'agent_cr_l2']), key=lambda kv: -len(kv[1])):
    print(f'\n{"=" * 100}\nagent CR: {_l1} / {_l2}  ({len(_g)} tickets)\n{"=" * 100}')
    for _, _r in _g.sample(n=min(4, len(_g)), random_state=0).iterrows():
        print(f'  [{_r[ID_COL]}] {_r["incident_subject"][:90]!r}')
        print(f'      customer: {_r["first_customer_msg"][:260]!r}')

### The curated taxonomies

Written by hand from the drafts, the pass-1 counts and the reading above. **Order fixes the codes**
(`EES1.1`, `ECR1.1`, ...): append, never insert. The last entry of each is the catch-all.
`SUBCATEGORIES = None` means the positives do not support sub-categories, and they keep the gate
label only.

In [ ]:
# Curated 2026-09-30 from the 40 first-pass positives. The discovery draft split them by admin
# action (creation / modification / deletion / access) with overlapping leaves; reading the tickets
# gives six distinct problems. Three positives asked to change an AUTHOR email on a published paper -
# a content correction - and are excluded by the gate definition instead.
SUBCATEGORIES = {
    'New user registration or registration ID request': (
        'An organisation, account manager or user asks for Embase accounts to be created for named '
        'people: new registration IDs and passcodes, login credentials, adding a user, or how a new '
        'colleague gets an account. Includes the agent sending out registration credentials.'),
    'Remove, revoke or delete a user account': (
        'A user must lose access or their account must go: deactivate or revoke a user who has left, '
        'delete a user from an organisation account, or permanently delete an Elsevier account.'),
    'Change or transfer the email on an account or licence': (
        'The email address on an existing account is no longer valid or must change, or a licence '
        'or seat held under one email address must move to another person or address.'),
    'Verification or login-details email not received': (
        'An email the customer needs in order to register or sign in has not arrived: the Embase '
        'email verification notice or code, a password reset email or code, or promised login '
        'details or registration credentials.'),
    'Personal account not linked to the institution or organisation': (
        'The customer has a personal Elsevier account but it is not linked, or is wrongly linked, to '
        'the institution or organisation that holds the Embase access: redirected back to the start '
        'page, no institution found, linking a personal account broke institutional access, access '
        'via the organisation email or access code does not work, or users are now required to '
        'register a personal account under SSO or email-domain access.'),
    'Sign-in problem with registered credentials': (
        'The customer has registered credentials but cannot sign in with them - an error with an ID '
        'and passcode or email and password - or can sign in but no longer sees what belongs to the '
        'account, such as saved searches or email alerts.'),
    'Other email-account issue': (
        'An email-account or registration problem that fits none of the sub-categories above.'),
}

# Curated 2026-09-30 from discovery over the 734 first-pass negatives, a read of the tickets, and
# the agent contact reasons. Pass 1 echoed the five example names in the extraction prompt, so the
# draft was used for volume only; the splits below come from reading the tickets.
CONTACT_REASONS = {
    'Correction to a published record': (
        'A publisher, author or editor asks for the metadata of a specific existing record to be '
        'updated to match the published version: author names, affiliations or emails, the title '
        'including a RETRACTED prefix, the abstract, the DOI, the publication date or source '
        'details.'),
    'Indexing error or data quality problem': (
        'A user or colleague reports that records are indexed or rendered wrongly: missing or wrong '
        'Emtree, drug, device or brand-name terms, wrong keyword headings, garbled abstracts, '
        'typographical errors, duplicate records, or a suggestion for a new Emtree term.'),
    'Missing article or record not yet indexed': (
        'A specific article, issue or conference paper is not in Embase or not fully loaded yet, '
        'and the customer asks for it to be added or when it will appear.'),
    'Journal coverage, evaluation or indexing status': (
        'Questions about a JOURNAL rather than a record: applying or suggesting a journal for '
        'inclusion, the evaluation or indexing status of a journal, discontinued or cancelled '
        'titles, complaints about the quality or integrity of an indexed journal, or journal lists.'),
    'Organisation access set-up or entitlement': (
        'An administrator, librarian or account manager asks about access for an organisation: '
        'activating a subscription, trial or Embase AI for an account, IP ranges, SSO, OpenAthens or '
        'Shibboleth configuration, API or developer credentials, consolidating accounts, or whether '
        'an institution subscribes.'),
    'User cannot access Embase': (
        'An individual cannot get into Embase through their institution or subscription: access '
        'via IP, VPN, OpenAthens or campus network fails, access is blocked with an error code, or a '
        'trial or paid access does not open.'),
    'Error message or technical fault': (
        'Embase or Embase AI misbehaves: search errors, pages or record details that will not load, '
        'error codes, a cookie banner that cannot be accepted, or browser-specific failures.'),
    'Search, export or product feature question': (
        'How to do something in Embase: building searches and search strategies, sources and '
        'filters, exporting or emailing records, alerts, linking to full text, or how a feature '
        'works.'),
    'Feedback or feature request': (
        'The customer comments on the interface or asks for a change: complaints about the new '
        'results page or search bar behaviour, or requests for new options such as a citation style.'),
    'Usage report or usage data request': (
        'A request for Embase usage statistics or reports for an account, or questions about '
        'usage figures.'),
    'Order, purchase, renewal or invoice': (
        'Buying Embase or asking about prices and plans, orders and payment problems, invoices, '
        'renewals, subscription dates, terms, or cancelling a subscription.'),
    'Other / not enough information': (
        'The ticket carries no request that fits the categories above: a duplicate pointing to '
        'another incident, a thank-you, an auto-reply or spam, an agent-only note, or too little '
        'text to tell.'),
}

SUB_OTHER = 'Other email-account issue'
CR_OTHER = 'Other / not enough information'
MIN_POSITIVES_FOR_SUBCATS = 15
for _name, _tax, _other in (('SUBCATEGORIES', SUBCATEGORIES, SUB_OTHER),
                            ('CONTACT_REASONS', CONTACT_REASONS, CR_OTHER)):
    if _tax is None:
        print(f'{_name}: not set')
        continue
    assert list(_tax)[-1] == _other, f'{_name}: the catch-all {_other!r} must come last'
    for _l, _d in _tax.items():
        assert _l and _d and "'" not in _d, f'{_name}: bad row {_l!r}'
    print(f'{_name}: {len(_tax)} categories')
    for _l, _d in _tax.items():
        print(f'  - {_l}\n      {_d}')

In [ ]:
STOP_BEFORE_STAGES_B_C = False   # set False once both curated lists have been reviewed
if STOP_BEFORE_STAGES_B_C:
    raise SystemExit('STOP_BEFORE_STAGES_B_C is True - review SUBCATEGORIES and CONTACT_REASONS, '
                     'then set it to False.')
assert CONTACT_REASONS is not None, 'CONTACT_REASONS must be curated before stage C'
print('Continuing to the stage B / C build.')

## 9. Build the curated modules, check the prompts, run stages B and C

In [ ]:
_specs = [module_spec(
    MOD_C, order=3, taxonomy=list(CONTACT_REASONS.items()),
    notes='Curated from discovery over the stage-A negatives.',
    what='contact reason', alt='the closest alternative contact reason',
    description=(
        'Classify why the customer contacted Embase support into exactly one contact reason. '
        f'{G_THREAD} {G_WHO.split(". Agents")[0]}. {G_FORWARD} {G_ONE} '
        f'Assign the single best-fitting LEAF code (e.g. {PREFIX_C}1.1). Never return an empty '
        f'code - if nothing else fits, or the ticket says too little, use {CR_OTHER}.'))]
if SUBCATEGORIES is not None:
    _specs.insert(0, module_spec(
        MOD_S, order=2, taxonomy=list(SUBCATEGORIES.items()),
        notes='Curated from discovery over the stage-A positives.',
        what='sub-category', alt='the closest alternative sub-category',
        description=(
            'Classify the email-account problem in this Embase support ticket into exactly one '
            'sub-category. The ticket is already known to be about email registration or an email '
            f'address on an account. {G_THREAD} {G_WHO} {G_FORWARD} {G_ONE} '
            f'Assign the single best-fitting LEAF code (e.g. {PREFIX_S}1.1). Never return an empty '
            f'code - if nothing else fits, use {SUB_OTHER}.')))

BUILT_MODULES = write_definition(_specs)
build(BUILT_MODULES)
assert leaf_labels(MOD_G) == GATE_LABELS, 'the gate taxonomy changed'
CR_LABELS = leaf_labels(MOD_C)
assert list(CR_LABELS.values()) == list(CONTACT_REASONS)
SUB_LABELS = leaf_labels(MOD_S) if SUBCATEGORIES is not None else {}
if SUBCATEGORIES is not None:
    assert list(SUB_LABELS.values()) == list(SUBCATEGORIES)

render_prompts(BUILT_MODULES)
for _mod, _tax in ((MOD_C, CONTACT_REASONS), (MOD_S, SUBCATEGORIES or {})):
    for _l, _d in _tax.items():
        assert _l in prompts[_mod] and _d in prompts[_mod], f'{_mod}: {_l} missing from the prompt'
print('\nCurated prompts verified.')

In [ ]:
print('=' * 78)
print(f'FULL CONTACT REASON SYSTEM PROMPT  ({MOD_C})')
print('=' * 78)
print(prompts[MOD_C])

In [ ]:
sub = pd.DataFrame(columns=[ID_COL, 'email_reg_subcategory'])
if SUBCATEGORIES is not None:
    sub = read_labels(run_stage(PROFILE_S, module=MOD_S,
                                label=f'FULL {MOD_S} - {len(positives)} tickets'),
                      STUB_S, SUB_LABELS, 'email_reg_subcategory')
    assert set(sub[ID_COL]) == set(positives[ID_COL])
    assert (sub['email_reg_subcategory'] != '').all(), 'blank or off-taxonomy sub-category codes'
    print('\n' + sub['email_reg_subcategory'].value_counts().to_string())

In [ ]:
cr = read_labels(run_stage(PROFILE_C, module=MOD_C, label=f'FULL {MOD_C} - {len(negatives)} tickets'),
                 STUB_C, CR_LABELS, 'main_contact_reason')
assert set(cr[ID_COL]) == set(negatives[ID_COL]), 'stage C returned a different id set'
print(f'\nBlank or off-taxonomy codes: {int((cr["main_contact_reason"] == "").sum())}')
print(cr['main_contact_reason'].value_counts().to_string())

# Read against the agent's own contact reason - agreement is not expected category-for-category,
# but a derived reason spread evenly across unrelated agent reasons is a definition problem.
_x = cr.merge(tickets[[ID_COL, 'INCIDENT_ID']], on=ID_COL).merge(meta, on='INCIDENT_ID', how='left')
print('\nDerived contact reason x agent contact reason (lvl1):')
print(pd.crosstab(_x['main_contact_reason'], _x['agent_cr_l1']).to_string())

## 10. Merge and write the deliverable

`final_tag` is the one column to report on: `Email registration: <sub-category>` for the positives
(or the gate label alone when there are no sub-categories), the derived contact reason otherwise.
The agent's contact reason from the metadata file sits alongside for reference.

In [ ]:
final = tickets.merge(sub, on=ID_COL, how='left', validate='1:1') \
               .merge(cr, on=ID_COL, how='left', validate='1:1') \
               .merge(meta, on='INCIDENT_ID', how='left', validate='1:1')
assert len(final) == len(tickets)
for _c in ('email_reg_subcategory', 'main_contact_reason'):
    final[_c] = final[_c].fillna('')
_pos = final['is_email_registration']
assert (final.loc[_pos, 'main_contact_reason'] == '').all()
assert (final.loc[~_pos, 'email_reg_subcategory'] == '').all()

final['final_tag'] = final['main_contact_reason']
final.loc[_pos, 'final_tag'] = ('Email registration: ' + final.loc[_pos, 'email_reg_subcategory']) \
    .where(final.loc[_pos, 'email_reg_subcategory'] != '', G_YES)
_orphan = final['final_tag'] == ''
if _orphan.any():
    print(f'WARNING: {int(_orphan.sum())} negatives came back with an off-taxonomy contact reason code')
    final.loc[_orphan, 'final_tag'] = '(unclassified - off-taxonomy code)'
print(final['final_tag'].value_counts().to_string())

In [ ]:
XL_LIMIT = 32_767
XL_MARKER = '  ...[TRUNCATED FOR EXCEL - see the companion CSV]'
OUT_COLS = [ID_COL, 'INCIDENT_ID', 'channel', 'incident_type', 'first_message', 'last_message',
            'month', 'incident_subject', 'final_tag', 'is_email_registration',
            'gate', 'gate_justification', 'gate_evidence',
            'email_reg_subcategory', 'email_reg_subcategory_justification',
            'email_reg_subcategory_evidence',
            'main_contact_reason', 'main_contact_reason_justification', 'main_contact_reason_evidence',
            'agent_cr_l1', 'agent_cr_l2', 'agent_resolution', 'customer_role', 'status',
            'kw_email_reg', 'kw_term', 'n_messages', 'n_customer_msgs', 'n_logged_by_agent',
            'n_agent_msgs', 'conversation_chars', 'was_truncated', TEXT_COL]
deliverable = final[[c for c in OUT_COLS if c in final.columns]].copy()
deliverable.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    long = df[TEXT_COL].str.len() > XL_LIMIT
    df.loc[long, TEXT_COL] = df.loc[long, TEXT_COL].str[:XL_LIMIT - len(XL_MARKER)] + XL_MARKER
    return df


_vc = deliverable['final_tag'].value_counts()
summary = pd.DataFrame({'final_tag': _vc.index, 'tickets': _vc.values,
                        'share_%': (_vc.values / len(deliverable) * 100).round(1)})
by_channel = pd.crosstab(deliverable['final_tag'], deliverable['channel'], margins=True)
by_month = pd.crosstab(deliverable['final_tag'], deliverable['month'], margins=True)
vs_agent = pd.crosstab(deliverable['final_tag'],
                       deliverable['agent_cr_l1'] + ' / ' + deliverable['agent_cr_l2'])

taxonomy_sheet = pd.DataFrame(
    [{'stage': 'A', 'module': MOD_G, 'code': CODE_OF_GATE[l], 'label': l, 'definition': d}
     for l, d in GATE_DEFS.items()]
    + [{'stage': 'B', 'module': MOD_S, 'code': c, 'label': l, 'definition': SUBCATEGORIES[l]}
       for c, l in SUB_LABELS.items()]
    + [{'stage': 'C', 'module': MOD_C, 'code': c, 'label': l, 'definition': CONTACT_REASONS[l]}
       for c, l in CR_LABELS.items()])

method = pd.DataFrame([
    ('Source', f'{SRC_TSV.name}: {len(raw):,} message rows, {len(tickets)} tickets, '
               f'{msgs["ts"].min():%Y-%m-%d} to {msgs["ts"].max():%Y-%m-%d}.'),
    ('De-duplication', f'{_n_dup} exact duplicate rows dropped.'),
    ('Cleaning', 'HTML to text (stdlib parser), entities decoded, whitespace normalised, quoted '
                 'reply history cut from every message except the first of each ticket, pasted-back '
                 'earlier messages removed, email banners removed. See notebook section 2.'),
    ('Speakers', 'THREAD_ENTRY_TYPE: Customer = CUSTOMER, Staff Account = AGENT, Customer Proxy = '
                 'CUSTOMER (logged by agent).'),
    ('Stage A', f'{MOD_G} on every ticket: {int(_pos.sum())} email registration / email-account.'),
    ('Stage B', ('Sub-categories discovered from the positives and curated by hand.'
                 if SUBCATEGORIES else 'No sub-categories.')),
    ('Stage C', f'{MOD_C} on the {int((~_pos).sum())} negatives; taxonomy discovered from those '
                'conversations and curated by hand. Not the agent contact reason.'),
    ('Agent contact reason', f'agent_cr_l1 / agent_cr_l2 from {METADATA_CSV.name}, reference only.'),
    ('Keyword audit', 'kw_email_reg is a regex over the customer side, kept only as a cross-check.'),
    ('Accuracy', 'Not measured - there is no hand-labelled set for this export.'),
    ('Fingerprints', '; '.join(f'{k}={v}' for k, v in PROMPT_FINGERPRINTS.items())),
], columns=['topic', 'detail'])

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as xw:
    fit_excel(deliverable).to_excel(xw, sheet_name='Tickets', index=False)
    summary.to_excel(xw, sheet_name='Summary', index=False)
    by_channel.to_excel(xw, sheet_name='Tag x channel')
    by_month.to_excel(xw, sheet_name='Tag x month')
    vs_agent.to_excel(xw, sheet_name='Tag x agent CR')
    taxonomy_sheet.to_excel(xw, sheet_name='Taxonomy', index=False)
    method.to_excel(xw, sheet_name='Method', index=False)
print(f'Wrote {FINAL_XLSX.relative_to(BASE)} and {FINAL_CSV.relative_to(BASE)}')
print('\n' + summary.to_string(index=False))

## 11. Put the previous config back

The outputs are written; they do not depend on the config staying as it is.

In [ ]:
RESTORE_CONFIG = True
_cmd_line = next((ln.strip() for ln in RESTORE_TXT.read_text(encoding='utf-8').splitlines()
                  if ln.strip().startswith('rca build')), None)
if RESTORE_CONFIG and _cmd_line:
    _parts = _cmd_line.split()
    run_cmd([sys.executable, str(BASE / _parts[1]), *_parts[2:]], 'restore config')
    print(f'\nRestored: {list(json.loads((BASE / "generated" / "pipeline_config.json").read_text(encoding="utf-8"))["modules"])}')
else:
    print(f'Not restored. To do it by hand:\n    {_cmd_line}')